In [1]:
# Install required libraries
!pip install -q langchain langchain-google-genai langsmith 2> /dev/null

import os
from google.colab import userdata

# Setup Gemini & LangSmith Tracing
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_KEY')
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGCHAIN_API_KEY')
os.environ["LANGCHAIN_PROJECT"] = "Resume_Screening_System"

print("Environment and LangSmith Tracing Ready!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 1.0 MB/s eta 0:00:00
Environment and LangSmith Tracing Ready!


In [7]:
# 1 Job Description
job_description = """
Role: Data Scientist
Required Skills: Python, Machine Learning, SQL, LangChain, Deep Learning.
Experience: 2+ years building AI/ML models.
"""

# 3 Distinct Resumes
resumes = {
    "Strong": "Jane Doe. 3 years exp. Skills: Python, SQL, Machine Learning, Deep Learning, LangChain, NLP. Built 5 LLM apps.",
    "Average": "John Smith. 1.5 years exp. Skills: Python, Data Analysis, SQL, Scikit-learn. Basic knowledge of Machine Learning.",
    "Weak": "Alice Jones. Fresh Graduate. Skills: HTML, CSS, JavaScript, basic Python. Looking for frontend or data entry roles."
}

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Initialize LLM
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", temperature=0.1)

# Step 1: Skill Extraction Prompt
extract_prompt = PromptTemplate.from_template(
    """Extract the skills, tools, and years of experience from this resume.
    Resume: {resume}
    Output format: Skills: [...], Experience: [...]"""
)
extract_chain = extract_prompt | llm | StrOutputParser()

# Step 2: Matching, Scoring & Explanation Prompt
eval_prompt = PromptTemplate.from_template(
    """You are an expert tech recruiter. Compare the candidate's extracted profile with the Job Description.
    Do NOT assume skills not present in the profile.

    Job Description: {jd}
    Candidate Profile: {extracted_profile}

    Provide:
    1. Fit Score (0-100)
    2. Explanation (Why this score? Match vs Gaps)
    """
)
eval_chain = eval_prompt | llm | StrOutputParser()

# Combine into a single modular LCEL Pipeline
# Resume goes into extractor -> result goes into evaluator along with JD
screening_pipeline = (
    {"extracted_profile": extract_chain, "jd": RunnablePassthrough()}
    | eval_chain
)

In [9]:
# Run the pipeline for all 3 candidates
for candidate_type, resume_text in resumes.items():
    print(f"\n{'='*40}\nEvaluating {candidate_type} Candidate\n{'='*40}")

    # We pass the resume to the first chain via invoke, but we also need to pass the JD to the second.
    # To keep LCEL simple, we'll invoke step 1, then step 2.

    print("Step 1: Extracting Skills...")
    extracted_data = extract_chain.invoke({"resume": resume_text})
    print(extracted_data)

    print("\nStep 2: Scoring & Explanation...")
    final_evaluation = eval_chain.invoke({
        "jd": job_description,
        "extracted_profile": extracted_data
    })
    print(final_evaluation)


Evaluating Strong Candidate
Step 1: Extracting Skills...
Skills: [Python, SQL, Machine Learning, Deep Learning, LangChain, NLP]
Experience: [3 years]

Step 2: Scoring & Explanation...
### 1. Fit Score: 100/100

### 2. Explanation
*   **Skill Match:** The candidate possesses 100% of the required technical stack (Python, Machine Learning, SQL, LangChain, and Deep Learning). Additionally, the candidate brings an extra relevant skill (NLP), which adds value to the role.
*   **Experience Match:** The candidate has 3 years of experience, which exceeds the minimum requirement of 2+ years.
*   **Conclusion:** This is an ideal candidate who meets all mandatory requirements and exceeds the experience threshold. No gaps were identified.

Evaluating Average Candidate
Step 1: Extracting Skills...
Skills: [Python, Data Analysis, SQL, Scikit-learn, Machine Learning]
Experience: [1.5 years]

Step 2: Scoring & Explanation...
### Candidate Evaluation Report

**1. Fit Score: 45/100**

**2. Explanation:*